In [ ]:
from sklearn.svm import SVR
kernels = ['linear','‘poly', 'rbf', 'sigmoid']
C_values = [0.2, 0.4, 0.6, 0.8, 1.0]
rbf_gamma = [0.05, 0.2, 0.4, 0.6]
ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
poly_degree = [1, 2, 3, 5]
train_set_size = 5000

In [ ]:
# Load the data

from pathlib import Path
import pandas as pd
import tarfile
import urllib.request
from sklearn.model_selection import train_test_split

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing_full = load_housing_data()


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, StandardScaler


train_set, test_set = train_test_split(housing_full, test_size=0.2,
                                       random_state=2911)
train_set = train_set[:train_set_size]

train_set.head(8)
train_labels = train_set["median_house_value"].copy()
train_set.drop("median_house_value", axis=1)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
14333,-117.91,33.76,22.0,7531.0,1569.0,5254.0,1523.0,3.8506,<1H OCEAN
11711,-117.31,34.15,7.0,5747.0,1307.0,2578.0,1147.0,3.3281,INLAND
2424,-121.92,36.62,52.0,1410.0,303.0,578.0,285.0,2.5625,NEAR OCEAN
1972,-118.21,34.64,16.0,2573.0,427.0,1273.0,426.0,5.9508,INLAND
14697,-117.99,33.92,27.0,5805.0,1152.0,3106.0,1144.0,4.0610,<1H OCEAN
...,...,...,...,...,...,...,...,...,...
13126,-122.43,37.74,52.0,2229.0,498.0,1079.0,472.0,5.0196,NEAR BAY
6432,-121.89,37.49,9.0,4909.0,577.0,1981.0,591.0,9.7194,<1H OCEAN
181,-118.10,34.01,29.0,2077.0,564.0,2087.0,543.0,2.6600,<1H OCEAN
17835,-121.57,39.48,15.0,202.0,54.0,145.0,40.0,0.8252,INLAND


In [ ]:
# cat_encoder = OneHotEncoder(sparse_output=False)
# housing_cat = train_set[["ocean_proximity"]]

# housing_cat_1hot = pd.DataFrame(cat_encoder.fit_transform(housing_cat),
#                          columns=cat_encoder.get_feature_names_out(),
#                          index=housing_cat.index)
# # print(cat_encoder.feature_names_in_)
# # print(cat_encoder.get_feature_names_out())


# # Drop the original 'ocean_proximity' column from train_set
# train_set_numerical = train_set.drop("ocean_proximity", axis=1)

In [ ]:
# Concatenate the numerical features with the one-hot encoded features
# train_set = pd.concat([train_set_numerical, housing_cat_1hot], axis=1)
# print(train_set.info())
# print(train_set.describe())
# std_scaler = StandardScaler()
# std_scaler.set_output(transform="pandas")

# housing_num_std_scaled = std_scaler.fit_transform(train_set_numerical)
# housing_num_std_scaled = pd.concat([housing_num_std_scaled, housing_cat_1hot], axis = 1)
# housing_num_std_scaled.head(8)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.kernel_approximation import RBFSampler
from sklearn.metrics.pairwise import rbf_kernel
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  # always return self!

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # feature names out

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())




In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector, make_column_transformer

num_attribs = ["longitude", "latitude", "housing_median_age", "total_rooms",
               "total_bedrooms", "population", "households", "median_income"]
cat_attribs = ["ocean_proximity"]

cat_pipeline = Pipeline([
    ("impute_mf", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ])



num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])
preprocessing = ColumnTransformer([
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                               "households", "median_income"]),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder=default_num_pipeline)

# preprocessing.set_output(transform="pandas")
housing_prepared = preprocessing.fit_transform(train_set)
housing_prepared_fr = pd.DataFrame(housing_prepared, columns=preprocessing.get_feature_names_out(),index=train_set.index)
housing_prepared_fr.head(8)

,bedrooms__ratio,rooms_per_house__ratio,people_per_house__ratio,log__total_bedrooms,log__total_rooms,log__population,log__households,log__median_income,geo__Cluster 0 similarity,geo__Cluster 1 similarity,...,geo__Cluster 7 similarity,geo__Cluster 8 similarity,geo__Cluster 9 similarity,cat__ocean_proximity_<1H OCEAN,cat__ocean_proximity_INLAND,cat__ocean_proximity_ISLAND,cat__ocean_proximity_NEAR BAY,cat__ocean_proximity_NEAR OCEAN,remainder__housing_median_age,remainder__median_house_value
14333,-0.089005,-0.218208,0.014551,1.767371,1.712657,2.072976,1.808904,0.232343,2.719557e-31,8.549398e-01,...,7.075719e-03,1.861299e-02,1.735132e-12,1.0,0.0,0.0,0.0,0.0,-0.539380,-0.344438
11711,0.209575,-0.190099,-0.053746,1.517761,1.352801,1.110874,1.425769,-0.075854,6.971382e-32,4.242574e-01,...,7.672016e-04,1.489886e-02,1.752496e-13,0.0,1.0,0.0,0.0,0.0,-1.746838,-0.736230
2424,0.013547,-0.217127,-0.066218,-0.479285,-0.517479,-0.909602,-0.455793,-0.628346,1.388250e-07,1.226497e-09,...,4.557552e-04,2.099962e-05,5.865592e-01,0.0,0.0,0.0,0.0,1.0,1.875537,0.244984
1972,-0.752134,0.250895,-0.011668,-0.010610,0.283141,0.157338,0.087359,1.152317,3.111210e-25,6.647960e-01,...,5.098788e-02,3.083524e-01,3.744192e-09,0.0,1.0,0.0,0.0,0.0,-1.022363,-0.225687
14697,-0.243727,-0.162752,-0.027190,1.345300,1.366167,1.362653,1.422230,0.344779,5.037558e-30,9.405268e-01,...,1.273024e-02,3.681535e-02,9.634383e-12,1.0,0.0,0.0,0.0,0.0,-0.136894,0.134901
4277,-0.737390,-0.118084,-0.045494,-1.733374,-1.402876,-1.646666,-1.416356,0.377826,3.032267e-04,3.463534e-14,...,2.940796e-07,2.622504e-08,5.352078e-01,0.0,0.0,0.0,0.0,1.0,0.587581,1.044171
17413,-0.066579,0.187438,0.131028,0.386414,0.358073,1.091339,0.196984,0.431199,5.391825e-30,9.473547e-01,...,1.354482e-02,3.683376e-02,1.043892e-11,1.0,0.0,0.0,0.0,0.0,0.265592,0.123632
15425,-0.824604,0.267197,-0.008190,0.571831,0.888290,0.790600,0.693213,0.646100,5.262078e-06,1.307724e-09,...,6.785736e-05,1.461295e-04,5.180103e-01,0.0,1.0,0.0,0.0,0.0,-0.941866,-0.562871


# Select and train a model


In [ ]:
train_set.head(8)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
14333,-117.91,33.76,22.0,7531.0,1569.0,5254.0,1523.0,3.8506,167400.0,<1H OCEAN
11711,-117.31,34.15,7.0,5747.0,1307.0,2578.0,1147.0,3.3281,122200.0,INLAND
2424,-121.92,36.62,52.0,1410.0,303.0,578.0,285.0,2.5625,235400.0,NEAR OCEAN
1972,-118.21,34.64,16.0,2573.0,427.0,1273.0,426.0,5.9508,181100.0,INLAND
14697,-117.99,33.92,27.0,5805.0,1152.0,3106.0,1144.0,4.0610,222700.0,<1H OCEAN
4277,-122.42,37.66,36.0,725.0,121.0,335.0,140.0,4.1250,327600.0,NEAR OCEAN
17413,-118.01,33.91,32.0,2722.0,571.0,2541.0,462.0,4.2305,221400.0,<1H OCEAN
15425,-120.95,37.61,17.0,4054.0,654.0,2034.0,667.0,4.6833,142200.0,INLAND


In [ ]:
from sklearn.svm import SVR

def add_prefix(step_name, params: dict) -> dict:
    return {f"{step_name}__{k}": v for k, v in params.items()}




svr = SVR()
full_pipeline = Pipeline([
      ("preprocessing", preprocessing),
      ("svr", svr)
])
# full_pipeline.get_params()

In [ ]:
from sklearn.model_selection import GridSearchCV

# kernels = ['linear','‘poly', 'rbf', 'sigmoid']
# C_values = [0.2, 0.4, 0,6, 0.8, 1.0]
# rbf_gamma = [0.05, 0.2, 0.4, 0.6]
# ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
# poly_degree = [1, 2, 3, 5]


param_grid = [
   {
       "svr__kernel": ["linear"],
       "svr__C": C_values
   },
   {
      "svr__kernel": ["poly", "sigmoid"],
      "svr__coef0": ps_coef0,
      "svr__degree": poly_degree,
      "svr__C": C_values
   },
   {
       "svr__kernel": ["rbf"],
       "svr__gamma": rbf_gamma,
       "svr__C": C_values
   }
]

# param_grid = [add_prefix("svr", param) for param in base_grid]





grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(train_set, train_labels)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                                                     SimpleImputer(strategy='median')),
                                                                                    ('standardscaler',
                                                                                     StandardScaler())]),
                                                          transformers=[('bedrooms',
                                                                         Pipeline(steps=[('simpleimputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('functiontransformer',
                                                                                          FunctionTransformer(feature_names_out=<f...
                                                                         <sklearn.compose._column_transformer.make_column_selector object at 0x7a1659b39700>)])),
                                       ('svr', SVR())]),
             n_jobs=-1,
             param_grid=[{'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__kernel': ['linear']},
                         {'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__coef0': [0.0, 0.05, 0.1, 0.15, 0.2, 0.5],
                          'svr__degree': [1, 2, 3, 5],
                          'svr__kernel': ['poly', 'sigmoid']},
                         {'svr__C': [0.2, 0.4, 0.6, 0.8, 1.0],
                          'svr__gamma': [0.05, 0.2, 0.4, 0.6],
                          'svr__kernel': ['rbf']}],
             scoring='neg_mean_squared_error')

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results["rmse"] = np.sqrt(-results["mean_test_score"])

results.sort_values("rmse")[:10]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_svr__C,param_svr__kernel,param_svr__coef0,param_svr__degree,param_svr__gamma,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,rmse
4,1.459742,0.048784,0.272782,0.055496,1.0,linear,NaN,NaN,NaN,"{'svr__C': 1.0, 'svr__kernel': 'linear'}",-1.254927e+10,-1.300260e+10,-1.426050e+10,-1.274081e+10,-1.214233e+10,-1.293910e+10,7.178113e+08,1,113750.170743
3,1.446558,0.005391,0.239521,0.008043,0.8,linear,NaN,NaN,NaN,"{'svr__C': 0.8, 'svr__kernel': 'linear'}",-1.277437e+10,-1.324027e+10,-1.422390e+10,-1.296366e+10,-1.237786e+10,-1.311601e+10,6.209107e+08,2,114525.154722
2,1.896699,0.284300,0.312565,0.091135,0.6,linear,NaN,NaN,NaN,"{'svr__C': 0.6, 'svr__kernel': 'linear'}",-1.300314e+10,-1.347857e+10,-1.426035e+10,-1.318680e+10,-1.258422e+10,-1.330262e+10,5.598953e+08,3,115336.966867
1,1.447290,0.012558,0.242920,0.009601,0.4,linear,NaN,NaN,NaN,"{'svr__C': 0.4, 'svr__kernel': 'linear'}",-1.324124e+10,-1.372187e+10,-1.437212e+10,-1.342032e+10,-1.280647e+10,-1.351240e+10,5.221749e+08,4,116242.861033
0,1.588064,0.177347,0.247776,0.012760,0.2,linear,NaN,NaN,NaN,"{'svr__C': 0.2, 'svr__kernel': 'linear'}",-1.348019e+10,-1.396303e+10,-1.455456e+10,-1.365985e+10,-1.303122e+10,-1.373777e+10,5.078413e+08,5,117208.238889
205,1.810239,0.243140,0.310009,0.098963,1.0,poly,0.05,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.05, 'svr__degr...",-1.361889e+10,-1.410113e+10,-1.469652e+10,-1.380648e+10,-1.315716e+10,-1.387604e+10,5.121717e+08,6,117796.587008
229,1.509166,0.018195,0.277114,0.016961,1.0,poly,0.20,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.2, 'svr__degre...",-1.361889e+10,-1.410113e+10,-1.469652e+10,-1.380648e+10,-1.315716e+10,-1.387604e+10,5.121717e+08,7,117796.587008
237,1.494359,0.026706,0.260036,0.013864,1.0,poly,0.50,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.5, 'svr__degre...",-1.361889e+10,-1.410113e+10,-1.469652e+10,-1.380648e+10,-1.315716e+10,-1.387604e+10,5.121717e+08,8,117796.587008
213,1.936874,0.487544,0.322655,0.076967,1.0,poly,0.10,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.1, 'svr__degre...",-1.361889e+10,-1.410113e+10,-1.469652e+10,-1.380648e+10,-1.315716e+10,-1.387604e+10,5.121717e+08,9,117796.587008
197,1.500469,0.043177,0.265228,0.006401,1.0,poly,0.00,1.0,NaN,"{'svr__C': 1.0, 'svr__coef0': 0.0, 'svr__degre...",-1.361889e+10,-1.410113e+10,-1.469652e+10,-1.380648e+10,-1.315716e+10,-1.387604e+10,5.121717e+08,10,117796.587008


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVR
from scipy.stats import randint, rang

# kernels = ['linear','‘poly', 'rbf', 'sigmoid']
# C_values = [0.2, 0.4, 0,6, 0.8, 1.0]
# rbf_gamma = [0.05, 0.2, 0.4, 0.6]
# ps_coef0 = [0.0, 0.05, 0.1, 0.15, 0.2, 0.5]
# poly_degree = [1, 2, 3, 5]
C_dist = randrange(0.0, 1.0)
ps_coef0_dist = randrange(0.0, 1.0)
rbf_gamm_dist = randrange(0.0, 1.0)
param_distrs = [
   {
       "svr__kernel": ["linear"],
       "svr__C" : C_dist
   },
   {
      "svr__kernel": ["poly", "sigmoid"],
      "svr__coef0": ps_coef0_dist,
      "svr__degree": randint(1, 5),
      "svr__C": C_dist
   },
   {
       "svr__kernel": ["rbf"],
       "svr__gamma": rbf_gamma_dist,
       "svr__C": C_dist
   }
]

svr = SVR()

random_search = RandomizedSearchCV(svr, full_pipeline)

ImportError: cannot import name 'randrange' from 'scipy.stats' (/usr/local/lib/python3.12/dist-packages/scipy/stats/__init__.py)